# Comparing two revisions of a pipeline

A scenario with `revision=` runs the flow as it was at another commit, next to
the current code. `credit_pipeline.py`'s last commit added a flat $50
affordability fee to `ratio`; this notebook compares it against the commit
just before.

How an experiment runs: the runner feeds the **baseline** (the unchanged
flow) and every scenario the same records, one batch at a time, and calls the
hooks of each **metric** — `on_batch_end` with the flow's output, and, for a
metric with a `nodes` glob, `on_node_end` with what each matching node wrote.
Each scenario gets its own copy of every metric; `summarise()` turns it into
a typed summary, and a **plot** draws the summaries of the metric it names.

In [1]:
import polars as pl
from IPython.display import display
from decider.experiments import (Bars, Changes, Describe, Divergence, Experiment, Progress, RunStatus,
                                 ValueCounts)
from example_projects.experiments.credit_pipeline import build


def show(spec):
    """Draw a plot's Vega-Lite spec inline; JupyterLab and VS Code render it interactively."""
    display({"application/vnd.vegalite.v5+json": spec}, raw=True)


ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [250.0, 500.0, 700.0, 250.0],
})
df

shape: (4, 2)
┌────────┬───────┐
│ income ┆ debt  │
│ ---    ┆ ---   │
│ f64    ┆ f64   │
╞════════╪═══════╡
│ 1000.0 ┆ 250.0 │
│ 2000.0 ┆ 500.0 │
│ 1500.0 ┆ 700.0 │
│ 800.0  ┆ 250.0 │
└────────┴───────┘


In [2]:
import subprocess
import example_projects.experiments.credit_pipeline as credit_pipeline

# The commit that added the fee, so this keeps working as the repository moves on.
fee = subprocess.run(["git", "log", "-1", "--format=%h", "--", credit_pipeline.__file__],
                     capture_output=True, text=True, check=True).stdout.strip()

result = (Experiment("version-drift", pipeline, df, entry=ENTRY)
          .scenario("before_fee", revision=f"{fee}^")
          .metric(Divergence(column="approved"),
                  Divergence(name="ratio", column="ratio", nodes="credit/ratio"))
          .plot(Changes(metric="divergence"))
          .run())
assert result.status is RunStatus.COMPLETED
[(s.name, s.status.value, s.revision.resolved[:7] if s.revision else "live") for s in result.scenarios]

[('baseline', 'completed', 'live'), ('before_fee', 'completed', '59b9825')]


`divergence` compares the flow's output; `ratio` is a node metric — its
`nodes="credit/ratio"` makes it compare what that one step wrote, which the
output doesn't carry. The fee moves every ratio, but only one record's
approval.

In [3]:
before = result.summaries["divergence"]["before_fee"]
ratio = result.summaries["ratio"]["before_fee"]
(before.changed, before.samples), (ratio.changed, ratio.samples[:2])

((1, [{'row': 3, 'expected': False, 'actual': True}]), (4, [{'row': 0, 'expected': 0.3, 'actual': 0.25}, {'row': 1, 'expected': 0.275, 'actual': 0.25}]))


In [4]:
show(result.plots["changes"])

<VegaLite 5 chart>

Building the experiment imported the live pipeline, under the same module name
the other revision's checkout has. Python caches imports by name, so that
revision runs in a fresh subprocess, which never had the live code loaded; it
writes its batches to disk and the runner reads them back in step with the
baseline, so the metrics see both side by side.